In [11]:
# Run this cell first
import os
import sys

project_root = os.getcwd()
if os.path.basename(project_root) == "notebooks":
    project_root = os.path.dirname(project_root)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root: {project_root}")

Project root: c:\Users\Administrator\Desktop\ai_doc_classifier


In [10]:
# Imports
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import joblib
import os

from utils.preprocess import clean_text
import fitz  # PyMuPDF for PDF text extraction


def load_resume_texts(data_dir):
    texts, labels = [], []

    for filename in os.listdir(data_dir):
        if not filename.endswith(".pdf"):
            continue

        doc = fitz.open(os.path.join(data_dir, filename))
        text = " ".join(page.get_text() for page in doc)
        cleaned = clean_text(text)

        lower_name = filename.lower()
        if "tech" in lower_name:
            label = "Tech"
        elif "finance" in lower_name:
            label = "Finance"
        elif "health" in lower_name:
            label = "Healthcare"
        elif "education" in lower_name:
            label = "Education"
        else:
            label = "Unknown"

        texts.append(cleaned)
        labels.append(label)

    return texts, labels


def train_resume_classifier(texts, labels):
    vectorizer = TfidfVectorizer(stop_words="english")
    X = vectorizer.fit_transform(texts)

    model = LogisticRegression(max_iter=1000)
    model.fit(X, labels)

    return model, vectorizer


def save_artifacts(model, vectorizer, models_dir):
    os.makedirs(models_dir, exist_ok=True)
    joblib.dump(model, os.path.join(models_dir, "resume_classifier.pkl"))
    joblib.dump(vectorizer, os.path.join(models_dir, "vectorizer.pkl"))


# Step 1: Load resumes from data/raw
data_dir = os.path.join(project_root, "data", "raw")
texts, labels = load_resume_texts(data_dir)

# Step 2: Train model
model, vectorizer = train_resume_classifier(texts, labels)

# Step 3: Save both model and vectorizer
models_dir = os.path.join(project_root, "models")
save_artifacts(model, vectorizer, models_dir)

print("Training complete. Model and vectorizer saved.")

# Step 4: Test predictions inside notebook
test_samples = [
    "Experienced Python developer with Django and Flask",
    "Financial analyst skilled in Excel and accounting",
    "Registered nurse with hospital experience",
    "Teacher with classroom management and curriculum design",
]

for sample in test_samples:
    pred = model.predict(vectorizer.transform([sample]))
    print(f"Text: {sample}\nPredicted Category: {pred[0]}\n")

# Step 5: Load saved artifacts for later reuse
models_dir = os.path.join(project_root, "models")
model = joblib.load(os.path.join(models_dir, "resume_classifier.pkl"))
vectorizer = joblib.load(os.path.join(models_dir, "vectorizer.pkl"))

print("Loaded saved model and vectorizer.")

Training complete. Model and vectorizer saved.
Text: Experienced Python developer with Django and Flask
Predicted Category: Tech

Text: Financial analyst skilled in Excel and accounting
Predicted Category: Finance

Text: Registered nurse with hospital experience
Predicted Category: Healthcare

Text: Teacher with classroom management and curriculum design
Predicted Category: Education

Loaded saved model and vectorizer.


In [12]:
# Reuse saved model and vectorizer for custom predictions
import joblib
import os


def load_saved_artifacts(models_dir=None):
    if models_dir is None:
        models_dir = os.path.join(project_root, "models")

    model = joblib.load(os.path.join(models_dir, "resume_classifier.pkl"))
    vectorizer = joblib.load(os.path.join(models_dir, "vectorizer.pkl"))
    return model, vectorizer


def predict_category(text, model=None, vectorizer=None):
    if model is None or vectorizer is None:
        model, vectorizer = load_saved_artifacts()

    return model.predict(vectorizer.transform([text]))[0]


custom_samples = [
    "Certified AWS engineer with cloud migration experience",
    "Budget analyst with forecasting and variance reporting",
    "Patient care coordinator in a community clinic",
    "Curriculum specialist focused on instructional design",
]

model, vectorizer = load_saved_artifacts()
for sample in custom_samples:
    print(f"Text: {sample}\nPredicted Category: {predict_category(sample, model, vectorizer)}\n")

Text: Certified AWS engineer with cloud migration experience
Predicted Category: Tech

Text: Budget analyst with forecasting and variance reporting
Predicted Category: Finance

Text: Patient care coordinator in a community clinic
Predicted Category: Healthcare

Text: Curriculum specialist focused on instructional design
Predicted Category: Education

